In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
import numpy as np

class AdaptiveWeightedFusion:
    def __init__(self):
        self.random_forest = GradientBoostingClassifier()
        self.naive_bayes = GaussianNB()
        self.svm = SVC(probability=True)
        self.scaler = MinMaxScaler()

    def fit(self, X_train, y_train, X_val, y_val):
        # Normalize features
        self.scaler.fit(X_train)
        X_train_scaled = self.scaler.transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)

        # Train classifiers
        self.random_forest.fit(X_train_scaled, y_train)
        self.naive_bayes.fit(X_train_scaled, y_train)
        self.svm.fit(X_train_scaled, y_train)

        # Calculate accuracies on validation set
        rf_acc = accuracy_score(y_val, self.random_forest.predict(X_val_scaled))
        nb_acc = accuracy_score(y_val, self.naive_bayes.predict(X_val_scaled))
        svm_acc = accuracy_score(y_val, self.svm.predict(X_val_scaled))

        # Calculate weights based on validation accuracies
        total_acc = rf_acc + nb_acc + svm_acc
        self.rf_weight = rf_acc / total_acc
        self.nb_weight = nb_acc / total_acc
        self.svm_weight = svm_acc / total_acc

    def predict(self, X):
        # Normalize features
        X_scaled = self.scaler.transform(X)

        # Predict probabilities from all classifiers
        rf_probs = self.random_forest.predict_proba(X_scaled)
        nb_probs = self.naive_bayes.predict_proba(X_scaled)
        svm_probs = self.svm.predict_proba(X_scaled)

        # Combine predictions using adaptive weights
        weighted_probs = (self.rf_weight * rf_probs) + (self.nb_weight * nb_probs) + (self.svm_weight * svm_probs)

        # Predict the class with the highest probability
        predicted_labels = np.argmax(weighted_probs, axis=1)

        # Map predicted labels to class names
        predicted_labels_str = np.where(predicted_labels == 1, 'Legit', 'Fraud')

        return predicted_labels_str

# Load the transaction dataset
data = pd.read_csv("/content/Firsttest.csv")  # Replace with the correct file path

# Select features and target variable
X = data[['Amount', 'Time', 'Domain']]
y = data['Classification']

# Convert 'Time' to seconds and add as a new column using .loc to avoid the SettingWithCopyWarning
X = X.copy()  # Add this line to explicitly create a copy of X
X.loc[:, 'Time_seconds'] = pd.to_timedelta(X['Time']).dt.total_seconds()
X = X.drop(columns=['Time'])

# Handle categorical feature 'Domain' using one-hot encoding
encoder = OneHotEncoder()
X_encoded = encoder.fit_transform(X[['Domain']]).toarray()

# Get names of encoded columns using get_feature_names_out
domain_columns = encoder.get_feature_names_out(['Domain'])

# Add encoded features to the dataset
X = X.drop(columns=['Domain'])
X = pd.concat([X, pd.DataFrame(X_encoded, columns=domain_columns)], axis=1)

print(X.columns)

# Split the data into training, validation, and testing sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE to the training data to balance the dataset
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Initialize the Adaptive Weighted Fusion classifier
awf = AdaptiveWeightedFusion()

# Train the classifier on the resampled training data and original validation data
awf.fit(X_train_resampled, y_train_resampled, X_val, y_val)

# Prompt user for transaction details for testing
amount = float(input("Enter transaction amount: "))
time = input("Enter transaction time (hh:mm:ss): ")
domain = input("Enter transaction domain: ")

# Convert user input time to seconds
time_seconds = pd.to_timedelta(time).total_seconds()

# Encode the domain input using the same encoder used for training
domain_encoded = encoder.transform([[domain]]).toarray()

# Create a DataFrame for the dummy transaction
dummy_transaction = pd.DataFrame({
    'Amount': [amount],
    'Time_seconds': [time_seconds]
})

# Add encoded domain columns to the dummy transaction
for i, col in enumerate(domain_columns):
    dummy_transaction[col] = domain_encoded[0][i]

# Reorder columns to match the order in X
dummy_transaction = dummy_transaction[X.columns]

# Make predictions on the dummy transaction
predicted_label = awf.predict(dummy_transaction)

# Display the predicted classification
print("Predicted Classification:", predicted_label[0])


In [ ]:
# Assuming you have already loaded your training data into the DataFrame X
print(X.columns)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Load the test set
# Assuming X_test and y_test are already defined from previous code

# Predict the labels for the test set
y_pred = awf.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Calculate precision
precision = precision_score(y_test, y_pred, pos_label='Legit')
print(f"Precision: {precision:.2f}")

# Calculate recall (sensitivity)
recall = recall_score(y_test, y_pred, pos_label='Legit')
print(f"Recall: {recall:.2f}")

# Calculate F1 score
f1 = f1_score(y_test, y_pred, pos_label='Legit')
print(f"F1 Score: {f1:.2f}")

# Calculate confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred, labels=['Fraud', 'Legit'])
print("Confusion Matrix:")
print(conf_matrix)

# Display the classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Fraud', 'Legit']))



In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.base import BaseEstimator
from sklearn.metrics import make_scorer, accuracy_score

# Custom estimator that wraps the AdaptiveWeightedFusion model to use with cross-validation
class AWFWrapper(BaseEstimator):
    def __init__(self):
        self.model = AdaptiveWeightedFusion()

    def fit(self, X, y):
        # Split the data into training and validation sets (70% training, 30% validation)
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

        # Initialize SMOTE and apply it to the training data
        smote = SMOTE(random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

        # Fit the model on the resampled training data and the original validation data
        self.model.fit(X_train_resampled, y_train_resampled, X_val, y_val)
        return self

    def predict(self, X):
        return self.model.predict(X)

# Create an instance of the AWFWrapper
awf_wrapper = AWFWrapper()

# Define a stratified k-fold cross-validation strategy
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define the scoring metric (accuracy)
scorer = make_scorer(accuracy_score)

# Perform cross-validation
cv_scores = cross_val_score(awf_wrapper, X, y, cv=skf, scoring=scorer)

# Calculate mean and standard deviation of the scores
mean_score = cv_scores.mean()
std_score = cv_scores.std()

# Print the cross-validation results
print(f"Cross-Validation Accuracy: {mean_score:.2f} (±{std_score:.2f})")
